In [2]:
pip install qiskit qiskit-aer

In [3]:
import qiskit
print(qiskit.__version__)

2.5.2


Import Required Libraries

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import qiskit
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

Exercise 1 Easy - Constant-Function Oracle for 3 Input Qubits & All-Zeros Verification

In [5]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
register_size = 3
# 3 query qubits + 1 target ancilla, 3 readout bits
circuit_c0 = QuantumCircuit(register_size + 1, register_size)
# State initialization: prepare query register in |+> and ancilla in |->
circuit_c0.x(register_size)
circuit_c0.h(range(register_size + 1))
circuit_c0.barrier()
# Constant-1 Oracle: f(x) = 1 (X gate on ancilla kicks back a global -1 phase)
circuit_c0.x(register_size)
circuit_c0.barrier()
# Interference stage: Hadamard transformation on input register
circuit_c0.h(range(register_size))
circuit_c0.measure(range(register_size), range(register_size))
engine = AerSimulator()
simulation_counts = engine.run(circuit_c0, shots=1024).result().get_counts()
print("--- Deutsch-Jozsa 3-Qubit Circuit (Constant-1) ---")
print(circuit_c0)
print("\nReadout Distribution:", simulation_counts)
eval_label = "Constant" if "000" in simulation_counts else "Balanced"
print(f"Function Determination: {eval_label} (Target state: 000)")

--- Deutsch-Jozsa 3-Qubit Circuit (Constant-1) ---
     ┌───┐      ░       ░ ┌───┐┌─┐      
q_0: ┤ H ├──────░───────░─┤ H ├┤M├──────
     ├───┤      ░       ░ ├───┤└╥┘┌─┐   
q_1: ┤ H ├──────░───────░─┤ H ├─╫─┤M├───
     ├───┤      ░       ░ ├───┤ ║ └╥┘┌─┐
q_2: ┤ H ├──────░───────░─┤ H ├─╫──╫─┤M├
     ├───┤┌───┐ ░ ┌───┐ ░ └───┘ ║  ║ └╥┘
q_3: ┤ X ├┤ H ├─░─┤ X ├─░───────╫──╫──╫─
     └───┘└───┘ ░ └───┘ ░       ║  ║  ║ 
c: 3/═══════════════════════════╩══╩══╩═
                                0  1  2 

Readout Distribution: {'000': 1024}
Function Determination: Constant (Target state: 000)


Exercise 2 Medium -
Balanced Parity Oracle for 3 Input Qubits & Non-Zero Verification

In [6]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
num_inputs = 3
circuit_bal = QuantumCircuit(num_inputs + 1, num_inputs)
# Prepare input states
circuit_bal.x(num_inputs)
circuit_bal.h(range(num_inputs + 1))
circuit_bal.barrier()
# Balanced Oracle: Alternating parity mapping f(x) = x0 ^ x2
circuit_bal.cx(0, num_inputs)
circuit_bal.cx(2, num_inputs)
circuit_bal.barrier()
# Interference and measurement
circuit_bal.h(range(num_inputs))
circuit_bal.measure(range(num_inputs), range(num_inputs))
counts_balanced = engine.run(circuit_bal, shots=1024).result().get_counts()
print("--- Deutsch-Jozsa 3-Qubit Circuit (Balanced Parity) ---")
print(circuit_bal)
print("\nReadout Distribution:", counts_balanced)
detected_bitstring = list(counts_balanced.keys())[0]
print(f"Function Determination: Balanced (Measured state: {detected_bitstring} != 000)")

--- Deutsch-Jozsa 3-Qubit Circuit (Balanced Parity) ---
     ┌───┐      ░            ░ ┌───┐┌─┐      
q_0: ┤ H ├──────░───■────────░─┤ H ├┤M├──────
     ├───┤      ░   │        ░ ├───┤└╥┘┌─┐   
q_1: ┤ H ├──────░───┼────────░─┤ H ├─╫─┤M├───
     ├───┤      ░   │        ░ ├───┤ ║ └╥┘┌─┐
q_2: ┤ H ├──────░───┼────■───░─┤ H ├─╫──╫─┤M├
     ├───┤┌───┐ ░ ┌─┴─┐┌─┴─┐ ░ └───┘ ║  ║ └╥┘
q_3: ┤ X ├┤ H ├─░─┤ X ├┤ X ├─░───────╫──╫──╫─
     └───┘└───┘ ░ └───┘└───┘ ░       ║  ║  ║ 
c: 3/════════════════════════════════╩══╩══╩═
                                     0  1  2 

Readout Distribution: {'101': 1024}
Function Determination: Balanced (Measured state: 101 != 000)


Exercise 3 Hard -
Generalized Implementation for Parameterized n (n = 2 to 5)

In [7]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
def construct_custom_oracle(width, category):
    subcircuit = QuantumCircuit(width + 1)
    if category == "constant":
        # Constant-0: no gates
        pass
    elif category == "balanced":
        # Balanced: CNOT from alternating qubits
        for idx in range(0, width, 2):
            subcircuit.cx(idx, width)
    return subcircuit
def test_dj_framework(width, category):
    qc = QuantumCircuit(width + 1, width)
    qc.x(width)
    qc.h(range(width + 1))
    qc.compose(construct_custom_oracle(width, category), inplace=True)
    qc.h(range(width))
    qc.measure(range(width), range(width))
    counts = AerSimulator().run(qc, shots=500).result().get_counts()
    sample = list(counts.keys())[0]
    classified_as = "Constant" if sample == "0" * width else "Balanced"
    return sample, classified_as
print("--- Scalable Deutsch-Jozsa Verification across Widths (n = 2..5) ---")
print(f"{'Width (n)':<12}{'Expected Type':<16}{'Measured String':<20}{'Outcome':<12}{'Check'}")
print("-" * 68)
for w in range(2, 6):
    for mode in ["constant", "balanced"]:
        bitstring, verdict = test_dj_framework(w, mode)
        verif = "CORRECT" if verdict.lower() == mode else "ERROR"
        print(f"{w:<12}{mode.capitalize():<16}{bitstring:<20}{verdict:<12}{verif}")

--- Scalable Deutsch-Jozsa Verification across Widths (n = 2..5) ---
Width (n)   Expected Type   Measured String     Outcome     Check
--------------------------------------------------------------------
2           Constant        00                  Constant    CORRECT
2           Balanced        01                  Balanced    CORRECT
3           Constant        000                 Constant    CORRECT
3           Balanced        101                 Balanced    CORRECT
4           Constant        0000                Constant    CORRECT
4           Balanced        0101                Balanced    CORRECT
5           Constant        00000               Constant    CORRECT
5           Balanced        10101               Balanced    CORRECT


Exercise 4 Real-world -
Classical vs. Quantum Query Complexity & Exponential Speedup

In [8]:
import numpy as np
# Classical vs Quantum complexity comparison
bits_array = np.arange(2, 11)
queries_classical = [int(2**(b - 1) + 1) for b in bits_array]
queries_quantum = [1 for _ in bits_array]
print("--- Exponential Separation in Query Complexity ---")
print(f"{'Input Bits (n)':<16}{'Classical Worst-Case':<25}{'Quantum Deutsch-Jozsa':<25}{'Speedup Factor'}")
print("=" * 80)
for idx, b in enumerate(bits_array):
    factor = queries_classical[idx] / queries_quantum[idx]
    print(f"{b:<16}{queries_classical[idx]:<25}{queries_quantum[idx]:<25}{factor:.1f}x")
print("=" * 80)

--- Exponential Separation in Query Complexity ---
Input Bits (n)  Classical Worst-Case     Quantum Deutsch-Jozsa    Speedup Factor
2               3                        1                        3.0x
3               5                        1                        5.0x
4               9                        1                        9.0x
5               17                       1                        17.0x
6               33                       1                        33.0x
7               65                       1                        65.0x
8               129                      1                        129.0x
9               257                      1                        257.0x
10              513                      1                        513.0x


Exercise 5 Challenge -
Randomized Balanced Oracle Generator & Algorithmic Robustness

In [9]:
import random
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
def generate_random_balanced_unitary(width):
    oracle_circuit = QuantumCircuit(width + 1, name="RndBalanced")
    # Generate random non-trivial bit pattern for parity projection
    active_bits = random.sample(range(width), k=random.randint(1, width))
    # Random output negation
    if random.choice([True, False]):
        oracle_circuit.x(width)
    for target in active_bits:
        oracle_circuit.cx(target, width)
    return oracle_circuit, active_bits
eval_width = 4
engine_sim = AerSimulator()
print("--- Testing Randomized Balanced Oracles (Width = 4) ---")
print(f"{'Iteration':<12}{'Active Qubits':<22}{'Measured Output':<20}{'Status'}")
print("-" * 65)
for iteration in range(1, 6):
    test_circ = QuantumCircuit(eval_width + 1, eval_width)
    test_circ.x(eval_width)
    test_circ.h(range(eval_width + 1))
    oracle_block, qubits_used = generate_random_balanced_unitary(eval_width)
    test_circ.compose(oracle_block, inplace=True)
    test_circ.h(range(eval_width))
    test_circ.measure(range(eval_width), range(eval_width))
    counts = engine_sim.run(test_circ, shots=500).result().get_counts()
    sample_key = list(counts.keys())[0]
    status_str = "SUCCESS (Balanced)" if sample_key != "0000" else "FAIL"
    print(f"{iteration:<12}{str(qubits_used):<22}{sample_key:<20}{status_str}")

--- Testing Randomized Balanced Oracles (Width = 4) ---
Iteration   Active Qubits         Measured Output     Status
-----------------------------------------------------------------
1           [1, 0]                0011                SUCCESS (Balanced)
2           [0, 3, 2]             1101                SUCCESS (Balanced)
3           [1]                   0010                SUCCESS (Balanced)
4           [1, 0, 2, 3]          1111                SUCCESS (Balanced)
5           [1, 2]                0110                SUCCESS (Balanced)
